In [ ]:
# ------------------------------------------------------------
# DOWNLOAD MODEL + QWEN + VAE
# ------------------------------------------------------------

import os
import requests

def download_file(url, out_path):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    filename = os.path.basename(out_path)
    
    print(f"Mulai download: {filename}...")
    
    with requests.get(url, stream=True, allow_redirects=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024): # Unduh per 1 MB
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    
                    if total:
                        print(
                            f"\rProgress {filename}: {downloaded / 1024**3:.2f} / {total / 1024**3:.2f} GB", 
                            end=""
                        )
    print(f"\n✓ Selesai mengunduh {filename}")
    print(f"Ukuran file: {os.path.getsize(out_path) / 1024**3:.2f} GB\n")

BASE_HF_URL = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files"
BASE_OUT_DIR = "/kaggle/working/anima_models"

# URL Hugging Face & Jalur Output Kaggle
paket_download = [
    {
        "url": f"{BASE_HF_URL}/diffusion_models/anima-base-v1.0.safetensors?download=true",
        "out": f"{BASE_OUT_DIR}/diffusion_models/anima-base-v1.0.safetensors"
    },
    {
        "url": f"{BASE_HF_URL}/text_encoders/qwen_3_06b_base.safetensors?download=true",
        "out": f"{BASE_OUT_DIR}/text_encoders/qwen_3_06b_base.safetensors"
    },
    {
        "url": f"{BASE_HF_URL}/vae/qwen_image_vae.safetensors?download=true",
        "out": f"{BASE_OUT_DIR}/vae/qwen_image_vae.safetensors"
    }
]

for item in paket_download:
    download_file(item["url"], item["out"])

print("Sudah selesai Download fullnya")

In [ ]:
# ============================================================
# CEKK DATASET DI KAGGEL
# ============================================================

import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for f in files:
        print("   ", f)

In [ ]:
# ============================================================
# DATASET INNPUT -> DATASET FOLDER 
# ============================================================

import os
import shutil

source_dir = "/kaggle/input/datasets/hthllh/__________"        #Ganti
output_dir = "/kaggle/working/dataset_training"

# Hapus dataset_training lama jika masih ada
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Buat folder baru
os.makedirs(output_dir, exist_ok=True)

# Copy semua file
for filename in os.listdir(source_dir):
    source = os.path.join(source_dir, filename)
    destination = os.path.join(output_dir, filename)

    if os.path.isfile(source):
        shutil.copy2(source, destination)

print("✓ Dataset berhasil disalin")
print()
print("Isi dataset_training:")

for filename in sorted(os.listdir(output_dir)):
    print(filename)

In [ ]:
# ============================================================
# CEKKK - VALIDASI DATASET
# ============================================================

import os

dataset_dir = "/kaggle/working/dataset_training"

images = [
    f for f in os.listdir(dataset_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
]

captions = [
    f for f in os.listdir(dataset_dir)
    if f.lower().endswith(".txt")
]

image_bases = {os.path.splitext(f)[0] for f in images}
caption_bases = {os.path.splitext(f)[0] for f in captions}

missing_caption = sorted(image_bases - caption_bases)
missing_image = sorted(caption_bases - image_bases)

print("=" * 50)
print("DATASET VALIDATION")
print("=" * 50)

print(f"Images   : {len(images)}")
print(f"Captions : {len(captions)}")
print()

if missing_caption:
    print("✗ Gambar tanpa caption:")
    print(missing_caption)
else:
    print("✓ Semua gambar memiliki caption")

if missing_image:
    print("✗ Caption tanpa gambar:")
    print(missing_image)
else:
    print("✓ Semua caption memiliki gambar")

print("=" * 50)

In [ ]:
# ============================================================
# CEK - MODEL ANIMA BASE + QWEN + VAE
# ============================================================

import os

MODEL_DIR = "/kaggle/working/anima_models"

print("=" * 60)
print("ANIMA MODELS DIRECTORY")
print("=" * 60)

if not os.path.exists(MODEL_DIR):
    print("✗ Folder tidak ditemukan:", MODEL_DIR)
else:
    for root, dirs, files in os.walk(MODEL_DIR):
        level = root.replace(MODEL_DIR, "").count(os.sep)
        indent = "  " * level

        print(f"{indent}{os.path.basename(root)}/")

        for f in files:
            path = os.path.join(root, f)
            size_mb = os.path.getsize(path) / (1024 ** 2)

            print(
                f"{indent}  {f} "
                f"({size_mb:.2f} MB)"
            )

In [ ]:
# ============================================================
# VALIDASI - MODEL ANIMA BASE + QWEN + VAE
# ============================================================

import os

MODEL_DIR = "/kaggle/working/anima_models"

MODELS = {
    "Anima Base v1.0":
        "anima-base-v1.0.safetensors",

    "Qwen3 0.6B":
        "qwen_3_06b_base.safetensors",

    "Qwen Image VAE":
        "qwen_image_vae.safetensors",
}

print("=" * 60)
print("FINAL ANIMA MODEL CHECK")
print("=" * 60)

all_ok = True

for name, filename in MODELS.items():
    path = os.path.join(MODEL_DIR, filename)

    print(f"\n{name}")
    print(f"  File   : {filename}")
    print(f"  Path   : {path}")

    if not os.path.isfile(path):
        print("  ✗ FILE TIDAK DITEMUKAN")
        all_ok = False
        continue

    size = os.path.getsize(path)
    size_mb = size / (1024 ** 2)
    size_gb = size / (1024 ** 3)

    print(f"  Size   : {size_mb:.2f} MB ({size_gb:.2f} GB)")

    if size_mb < 100:
        print("  ✗ UKURAN TIDAK VALID")
        all_ok = False
    else:
        print("  ✓ FILE VALID")

print("\n" + "=" * 60)

if all_ok:
    print("✓✓✓ SEMUA MODEL ANIMA SIAP ✓✓✓")
else:
    print("✗ MASIH ADA MODEL YANG BERMASALAH")

print("=" * 60)

In [ ]:
# ============================================================
# KOHYA SS ANIMA BASE
# ============================================================

import os
import subprocess

SD_SCRIPTS = "/kaggle/working/sd-scripts"

print("=" * 60)
print("CHECKING SD-SCRIPTS")
print("=" * 60)

if not os.path.exists(SD_SCRIPTS):
    print("sd-scripts belum ada.")
    print("Clone repository...")

    subprocess.run([
        "git", "clone",
        "https://github.com/kohya-ss/sd-scripts.git",
        SD_SCRIPTS
    ], check=True)

else:
    print("✓ sd-scripts directory sudah ada.")

print("\nChecking required Anima files...")

anima_train = os.path.join(
    SD_SCRIPTS,
    "anima_train_network.py"
)

lora_anima = os.path.join(
    SD_SCRIPTS,
    "networks",
    "lora_anima.py"
)

print(f"\nAnima training script:")
print(anima_train)
print("Exists:", os.path.exists(anima_train))

print(f"\nAnima LoRA network:")
print(lora_anima)
print("Exists:", os.path.exists(lora_anima))

print("\n" + "=" * 60)

if os.path.exists(anima_train) and os.path.exists(lora_anima):
    print("✓✓✓ SD-SCRIPTS SIAP UNTUK ANIMA ✓✓✓")
else:
    print("✗ File Anima belum lengkap.")
    print("JANGAN training dulu.")

print("=" * 60)

In [ ]:
# ============================================================
# sd-scripts
# ============================================================

import os
import subprocess
import sys

SD_SCRIPTS = "/kaggle/working/sd-scripts"

print("=" * 60)
print("INSTALLING SD-SCRIPTS")
print("=" * 60)

os.chdir(SD_SCRIPTS)

print("Current directory:")
print(os.getcwd())

print("\nInstalling sd-scripts + dependencies...\n")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        "."
    ],
    check=True
)

print("\n" + "=" * 60)
print("✓ SD-SCRIPTS BERHASIL DIINSTALL")
print("=" * 60)

In [ ]:
# ============================================================
# voluptuous
# ============================================================

import sys
import subprocess

print("=" * 60)
print("INSTALLING MISSING DEPENDENCY")
print("=" * 60)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "voluptuous"
    ],
    check=True
)

print("✓ voluptuous berhasil diinstall")

In [ ]:
# ============================================================
# CEKK SEMUA ENVIRONMENT
# ============================================================

import sys
import os

SD_SCRIPTS = "/kaggle/working/sd-scripts"

if SD_SCRIPTS not in sys.path:
    sys.path.insert(0, SD_SCRIPTS)

os.chdir(SD_SCRIPTS)

print("=" * 60)
print("ANIMA IMPORT TEST")
print("=" * 60)

try:
    import torch
    print(f"✓ PyTorch       : {torch.__version__}")
    print(f"✓ CUDA available: {torch.cuda.is_available()}")

    import accelerate
    print(f"✓ Accelerate    : {accelerate.__version__}")

    import transformers
    print(f"✓ Transformers  : {transformers.__version__}")

    import networks.lora_anima
    print("✓ networks.lora_anima")

    print("\nTesting Anima training module...")

    import anima_train_network
    print("✓ anima_train_network")

    print("\n" + "=" * 60)
    print("✓✓✓ ANIMA ENVIRONMENT SIAP ✓✓✓")
    print("=" * 60)

except Exception as e:
    print("\n" + "=" * 60)
    print("✗ IMPORT TEST GAGAL")
    print("=" * 60)
    print(type(e).__name__, ":", e)
    raise

In [ ]:
# ============================================================
# ANIMA LORA — KAGGLE TRAINING CONFIGURATION CHECK
# ============================================================

import os
import math

# -----------------------------
# PATH
# -----------------------------
DATASET_DIR = "/kaggle/working/dataset_training"
MODEL_DIR = "/kaggle/working/anima_models"
OUTPUT_DIR = "/kaggle/working/lora_output"

DIT_MODEL = os.path.join(
    MODEL_DIR,
    "anima-base-v1.0.safetensors"
)

QWEN3_MODEL = os.path.join(
    MODEL_DIR,
    "qwen_3_06b_base.safetensors"
)

VAE_MODEL = os.path.join(
    MODEL_DIR,
    "qwen_image_vae.safetensors"
)

# -----------------------------
# TRAINING SETTINGS
# -----------------------------
PROJECT_NAME = "____________________"                    # GANTII

# Default 32/16, Opsi lain 32/32
NETWORK_DIM = 32
NETWORK_ALPHA = 16

# Default 1e-4, Opsi lain 5e-5 sampai 1e-5
LEARNING_RATE = 1e-4

MAX_TRAIN_EPOCHS = 4                           # Total file Safetensorts
REPEATS = 50                                   # Pengulangan

RESOLUTION = 1024
CAPTION_DROPOUT = 0.0

# OPTIMASI MULTI-GPU & HIGH VRAM
NUM_GPUS = 2                  # Menggunakan GPU 0 dan GPU 1
BATCH_SIZE_PER_GPU = 2       # Batch per GPU (Total Effective Batch Size = ... )
GRADIENT_ACCUMULATION = 1
CACHE_LATENTS = True          # Mempercepat iterasi VAE
HIGH_VRAM = False             # Menjaga model tetap di VRAM GPU

TOTAL_BATCH_SIZE = BATCH_SIZE_PER_GPU * NUM_GPUS

# -----------------------------
# CHECK DATASET
# -----------------------------
print("=" * 60)
print("ANIMA LORA TRAINING CONFIGURATION")
print("=" * 60)

print("\n[DATASET]")

images = [
    f for f in os.listdir(DATASET_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
]

captions = [
    f for f in os.listdir(DATASET_DIR)
    if f.lower().endswith(".txt")
]

print(f"Dataset : {DATASET_DIR}")
print(f"Images  : {len(images)}")
print(f"Captions: {len(captions)}")

if len(images) != len(captions):
    raise RuntimeError(
        f"Jumlah gambar ({len(images)}) dan caption ({len(captions)}) tidak cocok."
    )

print("✓ Dataset cocok")

# -----------------------------
# CHECK MODELS
# -----------------------------
print("\n[MODELS]")

models = {
    "Anima Base v1.0": DIT_MODEL,
    "Qwen3 0.6B": QWEN3_MODEL,
    "Qwen Image VAE": VAE_MODEL
}

for name, path in models.items():

    exists = os.path.isfile(path)

    print(f"\n{name}")
    print(f"Path   : {path}")
    print(f"Exists : {exists}")

    if not exists:
        raise FileNotFoundError(f"Model tidak ditemukan: {path}")

    size_gb = os.path.getsize(path) / (1024 ** 3)

    print(f"Size   : {size_gb:.2f} GB")
    print("✓ OK")

# -----------------------------
# STEP CALCULATION
# -----------------------------
steps_per_epoch = math.ceil(
    len(images) * REPEATS /
    (TOTAL_BATCH_SIZE * GRADIENT_ACCUMULATION) # <--- Ubah BATCH_SIZE menjadi TOTAL_BATCH_SIZE
)

total_steps = steps_per_epoch * MAX_TRAIN_EPOCHS

print("\n[TRAINING]")

print(f"Project           : {PROJECT_NAME}")
print(f"Resolution        : {RESOLUTION}")
print(f"Network Dim       : {NETWORK_DIM}")
print(f"Network Alpha     : {NETWORK_ALPHA}")
print(f"Learning Rate     : {LEARNING_RATE}")
print(f"Repeats           : {REPEATS}")
print(f"Epochs            : {MAX_TRAIN_EPOCHS}")
print(f"Batch Size        : {TOTAL_BATCH_SIZE}") # <--- BATCH_SIZE menjadi TOTAL_BATCH_SIZE
print(f"Caption Dropout   : {CAPTION_DROPOUT}")

print("\n[STEP ESTIMATE]")

print(f"Images            : {len(images)}")
print(f"Steps / epoch     : {steps_per_epoch}")
print(f"Total steps       : {total_steps}")

# -----------------------------
# OUTPUT
# -----------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n[OUTPUT]")
print(f"Output directory  : {OUTPUT_DIR}")

print("\n" + "=" * 60)
print("✓✓✓ CONFIGURATION VALID ✓✓✓")
print("=" * 60)

print("\nGPU yang akan kita targetkan: GPU 0 & 1")
print("Belum menjalankan training.")

In [ ]:
# ============================================================
# ANIMA LORA — GENERATE TRAINING CONFIG (FIXED)
# ============================================================

import os
import toml
from datetime import datetime

CONFIG_DIR = "/kaggle/working/lora_configs"
os.makedirs(CONFIG_DIR, exist_ok=True)

CONFIG_PATH = os.path.join(
    CONFIG_DIR,
    f"{PROJECT_NAME}_training.toml"
)

training_config = {
    "pretrained_model_name_or_path": DIT_MODEL,

    "qwen3": QWEN3_MODEL,
    "vae": VAE_MODEL,

    "network_module": "networks.lora_anima",
    "network_dim": NETWORK_DIM,
    "network_alpha": NETWORK_ALPHA,
    "network_train_unet_only": False,

    "learning_rate": LEARNING_RATE,

    "optimizer_type": "AdamW",     # Kaggle GPU T4 15Gb x 2, AdamW - AdamW8Bit - AdaFactor
    "optimizer_args": [
        "weight_decay=0.1",
        "betas=[0.9, 0.99]"
    ],

    "lr_scheduler": "cosine",
    "lr_warmup_steps": 100,

    "max_train_epochs": MAX_TRAIN_EPOCHS,

    "train_batch_size": BATCH_SIZE_PER_GPU,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION,

    "max_grad_norm": 1.0,
    "seed": 42,

    "timestep_sampling": "sigmoid",
    "discrete_flow_shift": 1.0,

    "qwen3_max_token_length": 512,
    "t5_max_token_length": 512,

    "mixed_precision": "fp16",          # Pastikan tetap fp16, jangan bf16
    "gradient_checkpointing": True,

    "cache_latents": True,
    "cache_text_encoder_outputs": False,     # False

    "vae_chunk_size": 64,
    "vae_disable_cache": False,  

    "output_dir": "/kaggle/working/lora_output",
    "output_name": PROJECT_NAME,

    "save_model_as": "safetensors",
    "save_precision": "fp16",

    "save_every_n_epochs": 1,
    "save_last_n_epochs": 5,

    "shuffle_caption": True,
    "keep_tokens": 1,
    "caption_extension": ".txt",

    "noise_offset": 0.0,

    "training_comment": (
        f"Anima Base v1.0 LoRA - "
        f"{datetime.now().strftime('%Y-%m-%d')}"
    )
}

with open(CONFIG_PATH, "w") as f:
    toml.dump(training_config, f)

print("=" * 60)
print("ANIMA TRAINING CONFIG CREATED")
print("=" * 60)

print(f"\nConfig file:")
print(CONFIG_PATH)

# ============================================================
# ANIMA LORA — GENERATE DATASET CONFIG
# ============================================================

DATASET_CONFIG_PATH = os.path.join(
    CONFIG_DIR,
    f"{PROJECT_NAME}_dataset.toml"
)

dataset_config = {
    "general": {
        "resolution": RESOLUTION,
        "enable_bucket": True,
        "bucket_no_upscale": False,
        "bucket_reso_steps": 64,
        "min_bucket_reso": 512,
        "max_bucket_reso": 1536,
    },

    "datasets": [
        {
            "resolution": RESOLUTION,

            "subsets": [
                {
                    "num_repeats": REPEATS,
                    "image_dir": DATASET_DIR,
                    "caption_extension": ".txt",
                    "caption_dropout_rate": CAPTION_DROPOUT,
                }
            ]
        }
    ]
}

with open(DATASET_CONFIG_PATH, "w") as f:
    toml.dump(dataset_config, f)

print("\n" + "=" * 60)
print("ANIMA DATASET CONFIG CREATED")
print("=" * 60)
print(f"Config file: {DATASET_CONFIG_PATH}")
print("✓ Seluruh file TOML berhasil dibuat dengan path yang benar.")

In [ ]:
# ============================================================
# ISI LORA CONFIG
# ============================================================

import os

CONFIG_DIR = "/kaggle/working/lora_configs"

print("=" * 60)
print("ISI LORA CONFIGS")
print("=" * 60)

for filename in sorted(os.listdir(CONFIG_DIR)):
    path = os.path.join(CONFIG_DIR, filename)

    if os.path.isfile(path):
        size = os.path.getsize(path)

        print(f"{filename:<40} {size:,} bytes")

In [ ]:
# ============================================================
# DAFTAR NAMA CONFIG
# ============================================================

import os

CONFIG_DIR = "/kaggle/working/lora_configs"

print("=" * 60)
print("DAFTAR CONFIG FILE")
print("=" * 60)

if not os.path.exists(CONFIG_DIR):
    print("✗ Folder tidak ditemukan:", CONFIG_DIR)
else:
    files = sorted(os.listdir(CONFIG_DIR))

    for filename in files:
        path = os.path.join(CONFIG_DIR, filename)

        if os.path.isfile(path):
            print(filename)

print("=" * 60)

In [ ]:
# ============================================================
# ANIMA LORA — FINAL PRE-FLIGHT CHECK
# ============================================================

import os
import toml

print("=" * 60)
print("ANIMA LORA FINAL PRE-FLIGHT CHECK")
print("=" * 60)

errors = []

# ------------------------------------------------------------
# 1. Dataset
# ------------------------------------------------------------
print("\n[1] DATASET")

if os.path.isdir(DATASET_DIR):
    image_files = [
        f for f in os.listdir(DATASET_DIR)
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
    ]

    caption_files = [
        f for f in os.listdir(DATASET_DIR)
        if f.lower().endswith(".txt")
    ]

    print(f"  Path      : {DATASET_DIR}")
    print(f"  Images    : {len(image_files)}")
    print(f"  Captions  : {len(caption_files)}")

    if len(image_files) != len(caption_files):
        errors.append("Jumlah gambar dan caption tidak cocok.")
        print("  ✗ Image/caption mismatch")
    else:
        print("  ✓ Dataset OK")
else:
    errors.append("Dataset directory tidak ditemukan.")
    print("  ✗ Dataset directory missing")


# ------------------------------------------------------------
# 2. Models
# ------------------------------------------------------------
print("\n[2] MODELS")

models = {
    "Anima Base v1.0": DIT_MODEL,
    "Qwen3 0.6B": QWEN3_MODEL,
    "Qwen Image VAE": VAE_MODEL,
}

for name, path in models.items():
    exists = os.path.isfile(path)

    print(f"\n  {name}")
    print(f"    Path   : {path}")
    print(f"    Exists : {exists}")

    if exists:
        size_gb = os.path.getsize(path) / (1024 ** 3)
        print(f"    Size   : {size_gb:.2f} GB")
        print("    ✓ OK")
    else:
        errors.append(f"Model missing: {name}")
        print("    ✗ MISSING")


# ------------------------------------------------------------
# 3. sd-scripts
# ------------------------------------------------------------
print("\n[3] SD-SCRIPTS")

TRAIN_SCRIPT = "/kaggle/working/sd-scripts/anima_train_network.py"
LORA_MODULE = "/kaggle/working/sd-scripts/networks/lora_anima.py"

if os.path.isfile(TRAIN_SCRIPT):
    print("  ✓ anima_train_network.py")
else:
    errors.append("anima_train_network.py tidak ditemukan.")
    print("  ✗ anima_train_network.py")

if os.path.isfile(LORA_MODULE):
    print("  ✓ lora_anima.py")
else:
    errors.append("lora_anima.py tidak ditemukan.")
    print("  ✗ lora_anima.py")


# ------------------------------------------------------------
# 4. Config files
# ------------------------------------------------------------
print("\n[4] CONFIG FILES")

configs = {
    "Training config": CONFIG_PATH,
    "Dataset config": DATASET_CONFIG_PATH,
}

for name, path in configs.items():
    if os.path.isfile(path):
        print(f"  ✓ {name}")
        print(f"    {path}")
    else:
        errors.append(f"{name} missing.")
        print(f"  ✗ {name}")


# ------------------------------------------------------------
# 5. Read TOML configs
# ------------------------------------------------------------
print("\n[5] CONFIG VALIDATION")

try:
    train_cfg = toml.load(CONFIG_PATH)

    print("  ✓ Training TOML dapat dibaca")

    print("    pretrained_model_name_or_path:")
    print(f"      {train_cfg['pretrained_model_name_or_path']}")

    print("    network_module:")
    print(f"      {train_cfg['network_module']}")

    print("    network_dim:")
    print(f"      {train_cfg['network_dim']}")

except Exception as e:
    errors.append(f"Training TOML error: {e}")
    print(f"  ✗ Training TOML error: {e}")


try:
    dataset_cfg = toml.load(DATASET_CONFIG_PATH)

    print("  ✓ Dataset TOML dapat dibaca")

    subset = dataset_cfg["datasets"][0]["subsets"][0]

    print("    image_dir:")
    print(f"      {subset['image_dir']}")

    print("    num_repeats:")
    print(f"      {subset['num_repeats']}")

except Exception as e:
    errors.append(f"Dataset TOML error: {e}")
    print(f"  ✗ Dataset TOML error: {e}")


# ------------------------------------------------------------
# 6. Final result
# ------------------------------------------------------------
print("\n" + "=" * 60)

if errors:
    print("✗ PRE-FLIGHT CHECK GAGAL")
    print("=" * 60)

    for i, error in enumerate(errors, 1):
        print(f"{i}. {error}")

    print("\nJANGAN jalankan training dulu.")

else:
    print("✓✓✓ PRE-FLIGHT CHECK BERHASIL ✓✓✓")

    print("✓ Siap untuk tahap training.")

In [ ]:
# ============================================================
# INSTALL BITSANDBYTES
# ============================================================

import subprocess
import sys

print("=" * 60)
print("INSTALLING BITSANDBYTES")
print("=" * 60)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "bitsandbytes",
    ],
    check=True
)

print("\n✓ bitsandbytes berhasil diinstall")

In [ ]:
# ============================================================
# BITSANDBYTES TEST
# ============================================================

import torch
import bitsandbytes as bnb

print("=" * 60)
print("BITSANDBYTES CHECK")
print("=" * 60)

print("bitsandbytes :", bnb.__version__)
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

print("\n✓✓✓ BITSANDBYTES SIAP ✓✓✓")

In [ ]:
# ============================================================
# ANIMA LORA TRAINING CELL — 2× NVIDIA T4 (FIXED OUTPUT PATH)
# ============================================================

import os
import sys
import subprocess
import shlex
import time
import shutil

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------
SD_SCRIPTS_DIR = "/kaggle/working/sd-scripts"

DATASET_CONFIG = "/kaggle/working/lora_configs/____________________.toml"            #GANTI
TRAINING_CONFIG = "/kaggle/working/lora_configs/___________________.toml"          #GANTI

OUTPUT_DIR = "/kaggle/working/lora_output"

# ------------------------------------------------------------
# ENVIRONMENT
# ------------------------------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["TERM"] = "xterm"

# Pastikan folder output utama siap
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Pindah ke sd-scripts
os.chdir(SD_SCRIPTS_DIR)

print("=" * 60)
print("ANIMA LORA TRAINING — 2× T4")
print("=" * 60)

# ------------------------------------------------------------
# CHECK FILES
# ------------------------------------------------------------
required_files = [
    TRAINING_CONFIG,
    DATASET_CONFIG,
    os.path.join(SD_SCRIPTS_DIR, "anima_train_network.py"),
    os.path.join(SD_SCRIPTS_DIR, "networks", "lora_anima.py"),
]

print("\n[FILE CHECK]")
all_ok = True
for path in required_files:
    exists = os.path.exists(path)
    if exists:
        print(f"✓ {path}")
    else:
        print(f"✗ MISSING: {path}")
        all_ok = False

if not all_ok:
    raise RuntimeError("Ada file training yang belum ditemukan.")

# ------------------------------------------------------------
# GPU CHECK
# ------------------------------------------------------------
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Kurang dari 2 GPU terdeteksi.")

print("\n[GPU CHECK]")
for i in range(2):
    print(f"GPU {i} : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB)")

# ------------------------------------------------------------
# TRAINING COMMAND
# ------------------------------------------------------------
cmd = [
    sys.executable,
    "-m", "accelerate.commands.launch",
    "--num_processes", "2",
    "--num_machines", "1",
    "--mixed_precision", "fp16",
    os.path.join(SD_SCRIPTS_DIR, "anima_train_network.py"),
    "--config_file", TRAINING_CONFIG,
    "--dataset_config", DATASET_CONFIG,
]

print("\n[TRAINING COMMAND]")
print(" ".join(shlex.quote(x) for x in cmd))

print("\n" + "=" * 60)
print("START 2× GPU TRAINING")
print("=" * 60)

start_time = time.time()

process = subprocess.Popen(
    cmd,
    cwd=SD_SCRIPTS_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n\n⚠️ TRAINING DIHENTIKAN MANUAL.")
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
    raise
finally:
    if process.stdout:
        process.stdout.close()

return_code = process.wait()
elapsed = time.time() - start_time

print("\n" + "=" * 60)
print("TRAINING FINISHED")
print("=" * 60)
print(f"Exit code : {return_code}")
print(f"Elapsed   : {elapsed / 60:.2f} menit")

# ------------------------------------------------------------
# AUTO-SYNC FILE OUTPUT (Penyelamat File)
# ------------------------------------------------------------
if return_code == 0:
    print("\n[AUTO-SYNC OUTPUT]")
    
    # Cek folder lokal sd-scripts jika ada output tercecer
    fallback_sd_output = os.path.join(SD_SCRIPTS_DIR, "lora_output")
    if os.path.exists(fallback_sd_output):
        for f in os.listdir(fallback_sd_output):
            src = os.path.join(fallback_sd_output, f)
            dst = os.path.join(OUTPUT_DIR, f)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
                print(f"✓ Copied from sd-scripts: {f}")

    print("\nFile output di /kaggle/working/lora_output:")
    files = [f for f in os.listdir(OUTPUT_DIR) if os.path.isfile(os.path.join(OUTPUT_DIR, f))]
    
    if files:
        for filename in sorted(files):
            filepath = os.path.join(OUTPUT_DIR, filename)
            size_mb = os.path.getsize(filepath) / 1024**2
            print(f"  • {filename} ({size_mb:.2f} MB)")
            
        # Otomatis buat Zip agar langsung siap unduh
        zip_path = "/kaggle/working/____________________"         #Nama Zip saat sudah final trainingnya
        shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
        print(f"\n✓ BERHASIL MEMBUAT ZIP: {zip_path}.zip")
    else:
        print("⚠️ Folder output masih kosong. Periksa path `output_dir` di dalam file TOML.")

In [ ]:
# =====================================  WARNING =====================================
#                        BUAT HAPUS GAMBAR DATASET DAN OUTPUT LORA
# =====================================  WARNING =====================================

import os
import shutil

# Folder dataset lama
dataset_dir = "/kaggle/working/dataset_training"

# Hasil LoRA lama
output_dir = "/kaggle/working/lora_output"

# Configs LoRA lama
configs_dir = "/kaggle/working/lora_configs"

for folder in [dataset_dir, output_dir, configs_dir]:
    os.makedirs(folder, exist_ok=True)

    for name in os.listdir(folder):
        path = os.path.join(folder, name)

        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

    print(f"✓ Dikosongkan: {folder}")

print("\nSudah selesai hapus.")

In [ ]:
# ============================================================
# HAPUS HASIL ZIP SAAT INI
# ============================================================

import os
import glob

# Cari semua file .zip di direktori working
zip_files = glob.glob('/kaggle/working/*.zip')

# Hapus setiap file yang ditemukan
for file_path in zip_files:
    os.remove(file_path)
    print(f"Berhasil menghapus: {file_path}")


In [ ]:
import os

output_files = os.listdir('/kaggle/working/lora_output')
print("Isi folder lora_output:", output_files)

In [ ]:
import shutil

shutil.rmtree('/kaggle/working/dataset_training')